# 📚 Analyse des Mangas avec l'API Jikan

## 🎯 Objectif

L’objectif principal de ce projet est de recueillir, transformer et analyser des données de mangas via l’API Jikan. Les étapes incluent l’extraction de données brutes, leur préparation pour une analyse approfondie, et leur mise à disposition pour des recommandations ou des explorations futures.

Ce notebook vise à fournir :
- Une base de données propre et exploitable.
- Des insights détaillés sur les mangas, leurs auteurs et leurs caractéristiques.
- Une fondation solide pour la création d’un système de recommandations.

---

## 🗂️ Sommaire


1. [⚙️ 1. Préparation de l'Environnement](#Préparation-de-lEnvironnement)
   - 1.1 [📂 Importation des Bibliothèques](#Importation-des-Bibliothèques)
   - 1.2 [🔧 Paramètres Essentiels](#Paramètres-Essentiels)

2. [📈 2. Scraping des Données](#Scraping-des-Données)
   - 2.1 [🛠️ Fonction de Scraping des Données](#Fonction-de-Scraping-des-Données)
   - 2.2 [🛠️ Fonction de Scraping d'une Page](#Fonction-de-Scraping-dune-Page)
   - 2.3 [🚀 Exécution du Scraping Complet](#Exécution-du-Scraping)
   - 2.4 [🔗 Fusion des Fichiers JSON](#Fusion-des-Fichiers-JSON)
   - 2.5 [📜 Conclusion de la Phase de Scraping](#Conclusion-du-Scraping)

3. [📥 3. Chargement et Préparation des Données](#Chargement-des-Données)
   - 3.1 [📥 Chargement des Données dans un DataFrame](#Chargement-des-Données-dans-un-DataFrame)
   - 3.2 [📊 Analyse Descriptive du DataFrame](#Analyse-Descriptive-du-DataFrame)
   - 3.3 [🔧 Nettoyage, Filtrage et Simplification des Données](#Nettoyage-Filtrage-Simplification-des-Données)
   - 3.4 [🔄 Transformation des Types de Données et des Champs Textuels](#Transformation-Types-Champs-Textuels)
   - 3.5 [👤 Transformation des Auteurs](#Transformation-des-Auteurs)
   - 3.6 [🏷️ Transformation des Catégories](#Transformation-des-Catégories)
   - 3.7 [📊 Aperçu détaillé du DataFrame `manga`](#Aperçu-détaillé-du-DataFrame)

4. [📊 4. Analyse Exploratoire des Données](#Analyse-Exploratoire-des-Données)
   - 4.1 [📈 Analyse de la Popularité et Distribution des Scores](#Analyse-de-la-Popularité-et-Distribution-des-Scores)
   - 4.2 [📆 Fréquence des Publications par Année](#Fréquence-des-Publications-par-Année)
   - 4.3 [✍️ Contributions des Auteurs Individuels et Collectifs](#Contributions-des-Auteurs-Individuels-et-Collectifs)
   - 4.4 [💥 Analyse des Thèmes et Démographies](#Analyse-des-Thèmes-et-Démographies)
   - 4.5 [🔞 Proportion de Mangas NSFW](#Proportion-de-Mangas-NSFW)
   - 4.6 [🔍 Vérification des Types de Données et des Valeurs Manquantes](#Vérification-des-Types-de-Données-et-Valeurs-Manquantes)

5. [💾 5. Exportation des Données](#Exportation-des-Données)

6. [📜 Conclusion](#Conclusion)
---

## 📝 Introduction

Les mangas occupent une place importante dans la culture populaire mondiale. Grâce à l'API Jikan, nous avons accès à une riche base de données, permettant d’explorer des informations détaillées sur les mangas, leurs auteurs, et leurs genres.

Ce projet suit un pipeline structuré comprenant :
1. Le scraping des données depuis Jikan.
2. La préparation des données pour garantir leur qualité.
3. Une analyse exploratoire pour obtenir des insights utiles.
4. L'exportation des données prêtes pour des analyses ou projets futurs.

---

## ⚙️ 1. Préparation de l'Environnement <a name="Préparation-de-lEnvironnement"></a>

Dans cette section, nous configurons les bibliothèques et paramètres nécessaires pour lancer le projet. Cela inclut la définition des chemins de sauvegarde et des paramètres de scraping pour accéder aux données.


### 1.1 📂 Importation des Bibliothèques <a name="Importation-des-Bibliothèques"></a>

Nous importons ici les bibliothèques essentielles pour le scraping, la gestion des données et la visualisation.


In [1]:
import sys
from pathlib import Path

# Ajouter le dossier contenant config.py au chemin système
sys.path.append(str(Path("../src").resolve())) 

# Importer toutes les configurations depuis config.py
from config import *

# Récupérer le nom du notebook (sans extension .ipynb)
notebook_name = "01_Scraping_API_Jikan"  # Remplacez ceci par un moyen dynamique si nécessaire

# Initialiser les logs pour ce notebook
init_notebook_logs(notebook_name)

### 1.2 🔧 Paramètres Essentiels <a name="Paramètres-Essentiels"></a>

Les paramètres suivants sont définis pour assurer la gestion des données spécifiques à ce notebook :

- **URL de l'API** : Permet d'accéder aux données de mangas via l'API Jikan.
- **Chemin de Stockage des Données** : Les fichiers JSON temporaires et le fichier CSV final seront stockés dans le répertoire défini pour les données brutes (`raw_directory`), déjà configuré dans le fichier `config.py`.
- **Délai entre les Requêtes** : Un délai est ajouté entre chaque requête pour éviter de surcharger l'API et respecter les limitations d'accès.

Grâce à cette configuration, le notebook réutilise les chemins et paramètres globaux définis dans `config.py`, tout en ajoutant des ajustements spécifiques au contexte de l'analyse.


In [2]:
# Chemin de base pour les données
data_directory = CONFIG["raw_directory"]  # Utiliser directement le chemin défini dans config.py

# Configuration initiale spécifique au notebook
notebook_config = {
    "api_url": "https://api.jikan.moe/v4",  # URL de base de l'API
    "data_directory": data_directory,  # Chemin pour stocker les données brutes
    "database": "manga",  # Nom de la base de données à scraper
    "wait_time": 2,  # Délai en secondes entre les requêtes API
    "csv_output_file": "mangas_jikan"  # Fichier de sortie pour les données au format CSV
}

# Log de confirmation de la configuration initiale
logger.info(f"Configuration spécifique au notebook chargée avec succès : {notebook_config}")


## 📈 2. Scraping des Données <a name="Scraping-des-Données"></a>

Cette section détaille les étapes de scraping des données de mangas à partir de l'API Jikan, y compris la récupération, la gestion des erreurs et la fusion des fichiers JSON.


### 2.1 🛠️ Fonction de Scraping des Données : `fetch_data_from_jikan` <a name="Fonction-de-Scraping-des-Données"></a>

#### Description
La fonction `fetch_data_from_jikan` est utilisée pour interagir avec l'API Jikan afin de récupérer les données sur les mangas, page par page. Elle garantit une communication robuste avec l'API grâce à une gestion des erreurs intégrée et une limitation du nombre de requêtes pour respecter les règles de l'API.

In [3]:
def fetch_data_from_jikan(endpoint: str, page: int) -> Union[dict, None]:
    """
    Envoie une requête pour récupérer les données d'une page spécifique de l'API Jikan.

    Args:
        endpoint (str): Point de terminaison de l'API (e.g., 'manga').
        page (int): Numéro de la page à récupérer.

    Returns:
        Union[dict, None]: Données JSON récupérées si la requête réussit, sinon None.
    """
    # Construire l'URL pour la page spécifique de l'endpoint
    url = f"{notebook_config['api_url']}/{endpoint}?page={page}"
    max_attempts = 3  # Nombre maximal de tentatives autorisées

    # Tentatives de requêtes
    for attempt in range(1, max_attempts + 1):
        try:
            logger.info(f"[Tentative {attempt}/{max_attempts}] Récupération de la page {page} du endpoint '{endpoint}'")
            response = requests.get(url, timeout=10)  # Timeout pour éviter les blocages
            response.raise_for_status()  # Vérifie si le code de réponse est une erreur
            logger.info(f"Succès : Données récupérées pour la page {page} de '{endpoint}'")
            return response.json()
        
        except requests.exceptions.HTTPError as http_err:
            logger.error(f"Erreur HTTP pour la page {page} : {http_err}")
            if response.status_code == 429:  # Trop de requêtes
                logger.warning("Trop de requêtes. Attente de 10 secondes.")
                time.sleep(10)
            else:
                logger.warning("Nouvelle tentative après délai.")
                time.sleep(notebook_config["wait_time"])
        
        except requests.exceptions.Timeout:
            logger.error(f"Timeout atteint pour la page {page}. Nouvelle tentative dans {notebook_config['wait_time']} secondes.")
            time.sleep(notebook_config["wait_time"])
        
        except requests.RequestException as e:
            logger.error(f"Erreur de connexion pour la page {page} : {e}")
            time.sleep(notebook_config["wait_time"])

    logger.error(f"Échec de récupération des données pour la page {page} après {max_attempts} tentatives.")
    return None


#### Fonctionnalités principales :
- **Construction dynamique de l'URL** : Génère automatiquement l'URL en fonction du point de terminaison (`endpoint`) et du numéro de page (`page`).
- **Gestion des erreurs HTTP** : Capture et gère les erreurs courantes (ex. : dépassement de la limite de requêtes, timeout).
- **Requêtes sécurisées** : Utilise des tentatives répétées avec des pauses pour maximiser les chances de succès.
- **Temps d'attente configurable** : Définit des délais entre les requêtes pour éviter de surcharger l'API.

#### Arguments :
- `endpoint` *(str)* : Le point de terminaison de l'API (exemple : `'manga'`).
- `page` *(int)* : Le numéro de la page à récupérer.

#### Retour :
- *`dict`* : Les données JSON récupérées si la requête est réussie.
- *`None`* : Si toutes les tentatives échouent.

### 2.2 🛠️ Fonction de Scraping d'une Page : `scrape_page` <a name="Scraping-dune-Page"></a>

#### Description
La fonction `scrape_page` interagit avec l'API Jikan pour scraper les données d'une page spécifique et les enregistrer localement dans un fichier JSON. Elle est conçue pour garantir que chaque page est récupérée une seule fois, évitant les duplications inutiles, et pour gérer les erreurs potentielles lors du scraping ou de l'enregistrement des données.

In [4]:
def scrape_page(endpoint: str, page: int, file_path: Path) -> None:
    """
    Scrape une page spécifique de l'API et enregistre les données dans un fichier JSON
    si elles n'existent pas déjà.

    Args:
        endpoint (str): Point de terminaison de l'API pour récupérer les données.
        page (int): Numéro de la page à scraper.
        file_path (Path): Chemin du fichier où les données seront sauvegardées.
    """
    # Vérifier si le fichier existe déjà pour éviter un scraping redondant
    if file_path.exists():
        logger.info(f"[SKIP] Page {page} déjà présente dans {file_path}.")
        return  # Ne rien faire si le fichier existe déjà

    # Appel de la fonction fetch_data_from_jikan pour récupérer les données
    data = fetch_data_from_jikan(endpoint, page)
    if data:
        try:
            # Enregistrement des données dans un fichier JSON
            file_path.parent.mkdir(parents=True, exist_ok=True)  # Créer les répertoires si nécessaire
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data['data'], f, ensure_ascii=False, indent=4)
            logger.info(f"[SUCCESS] Données sauvegardées pour la page {page} dans {file_path}.")
        except IOError as e:
            # Gestion des erreurs d'écriture
            logger.error(f"[ERROR] Échec lors de la sauvegarde de la page {page} dans {file_path} : {e}")
    else:
        # Log en cas d'échec du scraping
        logger.warning(f"[FAIL] Aucune donnée récupérée pour la page {page}.")


#### Fonctionnalités principales :
- **Évitement des duplications** : La fonction vérifie si le fichier JSON existe déjà avant d’effectuer le scraping.
- **Gestion des erreurs** :
  - Capture les problèmes lors du scraping, tels que des erreurs HTTP ou des connexions échouées.
  - Gère les erreurs d'écriture sur disque pour assurer une robustesse maximale.
- **Création automatique des répertoires** : Les répertoires manquants sont créés automatiquement avant l'enregistrement des données.
- **Log des opérations** : Enregistre chaque étape du processus, y compris les succès, les duplications évitées, et les échecs.

#### Arguments :
- `endpoint` *(str)* : Le point de terminaison de l'API à interroger (exemple : `'manga'`).
- `page` *(int)* : Le numéro de la page à scraper.
- `file_path` *(Path)* : Le chemin complet où les données récupérées seront sauvegardées.

#### Retour :
- *`None`* : La fonction ne retourne aucune donnée, mais enregistre les résultats localement.


### 2.3 🚀 Exécution du Scraping de la Base de Données : `scrape_jikan_db` <a name="Exécution-du-Scraping"></a>

#### Description
La fonction `scrape_jikan_db` effectue le scraping complet d'une base de données spécifique (par exemple, "manga") via l'API Jikan. Elle récupère les données de chaque page disponible et les sauvegarde dans des fichiers JSON individuels. Cette approche facilite la gestion et l'organisation de grands ensembles de données.


In [5]:
def scrape_jikan_db(database: str) -> None:
    """
    Scrape toutes les pages d'une base de données spécifique de l'API Jikan
    et enregistre les données dans des fichiers JSON.

    Args:
        database (str): Nom de la base de données à scraper (ex : 'manga').
    """
    # Définir le répertoire pour stocker les fichiers de données de chaque page
    directory_path = notebook_config["data_directory"] / database
    directory_path.mkdir(parents=True, exist_ok=True)

    # Récupérer le nombre total de pages disponibles pour la base de données
    initial_data = fetch_data_from_jikan(database, page=1)
    
    if initial_data:
        last_page = initial_data.get('pagination', {}).get('last_visible_page', 0)
        logger.info(f"[START] Scraping de '{database}' démarré. Total de pages : {last_page}")
        
        if last_page > 0:
            for idx, page in enumerate(tqdm(range(1, last_page + 1), desc=f"Scraping {database}")):
                start_time = time.perf_counter()  # Mesurer le temps de récupération

                # Ajouter les métadonnées pour les logs
                is_last = (page == last_page)
                extra = {"index": idx + 1, "is_last": is_last}

                # Log avec métadonnées
                logger.bind(**extra).info(f"[{idx + 1}/{last_page}] Scraping de la page {page} de '{database}'")

                # Définir le chemin du fichier pour la page courante
                file_path = directory_path / f'page_{str(page).zfill(len(str(last_page)))}.json'

                # Scraper la page et sauvegarder les données
                scrape_page(database, page, file_path)

                # Pause adaptative pour respecter le délai défini
                elapsed_time = time.perf_counter() - start_time
                time.sleep(max(0, notebook_config["wait_time"] - elapsed_time))

            logger.bind(is_last=True).info(f"[FINISHED] Scraping terminé pour '{database}'.")
        else:
            logger.error(f"[ERROR] Aucune page disponible pour '{database}'.")
    else:
        logger.error(f"[FAIL] Impossible de récupérer les informations initiales pour '{database}'.")


#### Fonctionnalités principales :
- **Récupération des informations de pagination** : Identifie automatiquement le nombre total de pages à scraper pour la base de données sélectionnée.
- **Enregistrement structuré** : Les données de chaque page sont enregistrées dans un fichier JSON distinct, organisé dans un répertoire dédié.
- **Gestion des délais et performances** :
  - Pause adaptative entre les requêtes pour respecter les limitations de l'API.
  - Mesure du temps d'exécution pour optimiser le traitement.
- **Logs détaillés** : 
  - Indique le début, la progression, et la fin du processus de scraping.
  - Enregistre les erreurs ou anomalies rencontrées pendant l'exécution.

#### Arguments :
- `database` *(str)* : Le nom de la base de données à scraper (par exemple, `'manga'` ou `'anime'`).


### 2.4 🚀 Lancement du Scraping de la Base de Données <a name="Lancement-du-Scraping"></a>

En appelant `scrape_jikan_db(notebook_config['database'])`, nous démarrons le processus de scraping pour la base de données spécifiée dans notre configuration, ici `"manga"`. La fonction exécute les étapes suivantes :

#### Étapes principales :
- **Récupération de toutes les pages** :
  - La fonction itère sur toutes les pages disponibles de la base de données et envoie des requêtes à l'API Jikan.
  - Elle garantit la collecte complète des données.

- **Sauvegarde des données** :
  - Chaque page est enregistrée dans un fichier JSON distinct, stocké dans un répertoire dédié.
  - Cette approche permet une organisation claire et facilite le traitement ultérieur des données.

- **Respect des limitations de l'API** :
  - La fonction intègre des pauses entre les requêtes pour respecter les limitations d'accès de l'API.
  - Elle s'adapte en cas d'erreurs ou de dépassement du taux de requêtes.

In [6]:
# Lancer le scraping de la base de données spécifiée
scrape_jikan_db(notebook_config['database'])

Scraping manga: 100%|██████████| 2919/2919 [1:37:45<00:00,  2.01s/it]


### 2.5 🔗 Fusion et Centralisation des Fichiers JSON : `merge_files` <a name="Fusion-et-Centralisation-des-Fichiers-JSON"></a>

La fonction `merge_files` joue un rôle crucial dans la préparation des données collectées. Elle regroupe tous les fichiers JSON générés lors du scraping en un fichier unique, simplifiant ainsi les étapes suivantes de traitement et d’analyse.


In [7]:
def merge_files(database: str) -> str:
    """
    Fusionne tous les fichiers JSON d'un dossier en un seul fichier JSON et supprime le dossier temporaire.

    Args:
        database (str): Nom de la base de données correspondant au dossier contenant les fichiers JSON.

    Returns:
        str: Le chemin du fichier JSON fusionné, ou une chaîne vide en cas d'erreur.
    """
    # Chemin du dossier contenant les fichiers JSON à fusionner
    directory_path = notebook_config["data_directory"] / database
    merged_file_path = notebook_config["data_directory"] / f"{database}.json"

    try:
        # Vérification que le répertoire existe
        if not directory_path.exists():
            logger.error(f"Le dossier {directory_path} n'existe pas.")
            return ""

        data = []
        # Parcours des fichiers JSON dans le dossier
        for file_name in tqdm(os.listdir(directory_path), desc="Fusion des fichiers JSON"):
            file_path = directory_path / file_name
            # Vérifier si c'est bien un fichier JSON
            if file_path.suffix != ".json":
                logger.warning(f"Fichier ignoré (non JSON) : {file_path}")
                continue

            try:
                with open(file_path, 'r') as f:
                    data.extend(json.load(f))  # Ajoute les données de chaque fichier à la liste principale
            except json.JSONDecodeError as e:
                logger.error(f"Erreur lors de la lecture du fichier {file_path} : {e}")
            except Exception as e:
                logger.error(f"Erreur inattendue avec le fichier {file_path} : {e}")

        # Vérification si des données ont été collectées
        if not data:
            logger.error(f"Aucune donnée collectée à partir des fichiers dans {directory_path}.")
            return ""

        # Écriture des données fusionnées dans un fichier JSON unique
        with open(merged_file_path, 'w') as f:
            json.dump(data, f, indent=4)

        # Suppression du dossier contenant les fichiers JSON originaux pour libérer de l'espace
        shutil.rmtree(directory_path)
        logger.info(f"Fusion des fichiers terminée. Fichier créé : {merged_file_path}")

        return str(merged_file_path)
    except FileNotFoundError as e:
        logger.error(f"Fichier ou dossier introuvable : {e}")
        return ""
    except PermissionError as e:
        logger.error(f"Problème de permissions lors de la manipulation des fichiers : {e}")
        return ""
    except Exception as e:
        logger.error(f"Erreur lors de la fusion des fichiers JSON : {e}")
        return ""



#### **🔧 Fonctionnalités principales**

1. **Parcours des fichiers JSON** :
   - La fonction explore tous les fichiers JSON présents dans le répertoire correspondant à une base de données spécifique (ex. : `manga`).
   - Les données de chaque fichier sont extraites et combinées dans une seule structure.

2. **Fusion en un fichier unique** :
   - Toutes les données collectées sont regroupées dans un fichier JSON unique, ce qui facilite leur manipulation et leur analyse.

3. **Nettoyage des fichiers temporaires** :
   - Une fois les données fusionnées, le répertoire contenant les fichiers individuels est supprimé pour libérer de l’espace disque.


#### **💡 Pourquoi fusionner les fichiers JSON ?**

- **Facilite l’analyse** :
  - Avec un seul fichier JSON, le chargement des données dans des outils comme Pandas devient simple et rapide.
- **Organisation simplifiée** :
  - Centraliser les données réduit l’encombrement des répertoires et améliore la lisibilité de la structure des fichiers.
- **Optimisation de l’espace disque** :
  - En supprimant les fichiers individuels après fusion, l’espace utilisé est minimisé.

#### **⚠️ Points importants**

- **Validation des fichiers JSON** :
  - La fonction ignore les fichiers non conformes ou corrompus pour garantir l’intégrité des données fusionnées.
- **Logs détaillés** :
  - Chaque étape de la fusion est journalisée, facilitant le suivi et le débogage en cas de problème.
- **Fiabilité** :
  - En cas d’erreur, un log est généré et la fusion n’affecte pas les données collectées précédemment.



Les fichiers JSON, créés pour chaque page de données, sont ensuite fusionnés en un seul fichier pour simplifier le chargement.

In [8]:
# Lancement du processus de fusion
merged_file_path = merge_files(notebook_config['database'])

Fusion des fichiers JSON: 100%|██████████| 2919/2919 [00:02<00:00, 1035.15it/s]


### 2.5 📜 Conclusion de la Phase de Scraping <a name="Conclusion-du-Scraping"></a>

La phase de scraping a permis de collecter et de centraliser les données de manière efficace en suivant un processus structuré et robuste. Voici les étapes clés que nous avons accomplies :

---

#### **1. Configuration Initiale**
- Mise en place des paramètres essentiels, incluant :
  - Le chemin de sauvegarde des données.
  - Les paramètres d’accès à l’API Jikan.
  - Les limites de requêtes et les temps d’attente pour respecter les restrictions de l’API.

---

#### **2. Développement des Fonctions de Scraping**
- **`fetch_data_from_jikan`** :
  - Envoi de requêtes à l’API Jikan avec gestion des erreurs (codes HTTP, timeout).
  - Respect des limitations de l’API via des pauses adaptatives.
  
- **`scrape_page`** :
  - Récupération des données d’une page spécifique.
  - Sauvegarde des données dans des fichiers JSON tout en évitant les duplications.
  
- **`scrape_jikan_db`** :
  - Exécution du scraping complet pour une base de données donnée (ex. : `manga`).
  - Gestion des pauses entre les requêtes pour éviter de surcharger l’API.
  - Enregistrement des données dans des fichiers JSON distincts pour chaque page.

---

#### **3. Fusion des Fichiers JSON**
- Combinaison des fichiers JSON générés pour chaque page en un seul fichier unique.
- Nettoyage des fichiers temporaires pour optimiser l’espace disque.
- Résultat : un fichier JSON fusionné prêt à être chargé pour l’analyse.

---

### **Prochaines étapes**
Nous disposons maintenant de toutes les données nécessaires pour passer à la phase suivante : le **chargement**, le **nettoyage**, et la **préparation** des données en vue de leur analyse. Cette étape permettra d’extraire des insights significatifs et de répondre aux objectifs fixés pour ce projet.


## 📥 3. Chargement et Préparation des Données <a name="Chargement-des-Données"></a>

Après avoir fusionné les fichiers JSON, les données sont chargées dans un DataFrame pour un traitement plus approfondi. Cette étape inclut plusieurs opérations de nettoyage et de transformation pour préparer les données à une analyse plus détaillée. Parmi ces opérations, nous allons :

1. 📥 **Charger les données dans un DataFrame.**
2. 📊 **Trier les données pour une meilleure organisation.**
3. ❌ **Supprimer les doublons pour assurer l'unicité des enregistrements.**
4. 🔄 **Transformer certaines colonnes pour une meilleure lisibilité et utilisation.**
5. 🧹 **Nettoyer les champs textuels pour assurer la cohérence des données.**


### 3.1 📥 Chargement des Données dans un DataFrame <a name="Chargement-des-Données"></a>

Dans cette étape, nous chargeons les données fusionnées dans un DataFrame `Pandas` pour les préparer à une analyse ultérieure.

- 📊 **Pandas** : Nous utilisons `Pandas` pour manipuler les données dans un format structuré, ce qui permet d'effectuer des opérations d'analyse et de nettoyage facilement.
  
- 📝 **Gestion des erreurs avec Loguru** : Chaque tentative de chargement est enregistrée avec `Loguru` pour assurer un suivi en cas de problème. Cela permet de documenter les erreurs potentielles et d'identifier les problèmes lors du chargement des données.


In [9]:
# Chemin du fichier fusionné
merged_file_path = notebook_config["data_directory"] / f"{notebook_config['database']}.json"

# Chargement du fichier JSON dans un DataFrame Pandas
if merged_file_path.exists():
    try:
        manga = pd.read_json(merged_file_path)
        logger.info(f"Chargement réussi des données depuis {merged_file_path} dans un DataFrame Pandas.")
    except ValueError as ve:
        logger.error(f"Erreur de format dans le fichier JSON {merged_file_path} : {ve}")
        manga = pd.DataFrame()  # Initialise un DataFrame vide en cas d'erreur de format
    except Exception as e:
        logger.error(f"Erreur inattendue lors du chargement du fichier JSON {merged_file_path} : {e}")
        manga = pd.DataFrame()  # Initialise un DataFrame vide en cas d'erreur inattendue
else:
    logger.error(f"Fichier JSON fusionné introuvable : {merged_file_path}")
    manga = pd.DataFrame()  # Initialise un DataFrame vide si le fichier n'existe pas

# Vérification du DataFrame chargé
if manga.empty:
    logger.warning("Le DataFrame Pandas est vide. Vérifiez la source des données ou le fichier JSON.")
else:
    logger.info(f"Le DataFrame contient {len(manga)} lignes et {len(manga.columns)} colonnes.")


### 3.2 📊 Analyse Descriptive du DataFrame <a name="Analyse-Descriptive-du-DataFrame"></a>

   - 📏 **Statistiques de Base** : Dimensions, types de données, et aperçu des premières lignes.
   - ❓ **Valeurs Manquantes et Doublons** : Calcul des pourcentages pour chaque colonne.
   - 🔢 **Types de Données** : Vérification pour anticiper les transformations nécessaires.


La fonction `description_donnees` fournit un résumé descriptif du DataFrame en affichant des statistiques clés, notamment :

- 📏 **Le nombre de lignes et de colonnes.**
- ❓ **Le pourcentage de valeurs manquantes.**
- 🔁 **Le pourcentage de doublons.**
- 🔢 **La répartition des types de colonnes.**
- 💾 **La mémoire utilisée par le DataFrame.**

Cette analyse permet de mieux comprendre la structure générale des données, d’identifier les valeurs manquantes et les doublons, et d’évaluer les types de données présents pour guider les étapes de nettoyage et de transformation.

In [10]:
def description_donnees(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Analyse un DataFrame et retourne un résumé sous forme de DataFrame incluant diverses statistiques.
    
    Arguments:
    dataframe (pd.DataFrame) : Le DataFrame à analyser.
    
    Retourne:
    pd.DataFrame : Un DataFrame contenant le résumé des informations analysées.
    """
    # Vérifie les types de colonnes pour éviter les problèmes avec des types non pris en charge
    for col in dataframe.columns:
        if dataframe[col].apply(lambda x: isinstance(x, (list, dict))).any():
            dataframe[col] = dataframe[col].astype(str)
    
    # Calcul du nombre total de lignes et de colonnes dans le DataFrame
    nb_lignes, nb_colonnes = dataframe.shape

    # Calcul du pourcentage total de valeurs manquantes (NaN) dans le DataFrame
    pourcentage_nan = round(dataframe.isna().sum().sum() / dataframe.size * 100, 2)

    # Calcul du nombre de lignes en double
    nombre_doublons = dataframe.duplicated().sum()
    pourcentage_doublons = round(nombre_doublons / nb_lignes * 100, 2) if nb_lignes > 0 else 0

    # Comptage des types de colonnes dans le DataFrame
    nb_colonnes_objet = dataframe.select_dtypes(include=['object']).shape[1]
    nb_colonnes_flottant = dataframe.select_dtypes(include=['float']).shape[1]
    nb_colonnes_entier = dataframe.select_dtypes(include=['int']).shape[1]
    nb_colonnes_booleen = dataframe.select_dtypes(include=['bool']).shape[1]

    # Taille mémoire utilisée par le DataFrame en Mo
    taille_memoire = round(dataframe.memory_usage(deep=True).sum() / 1024**2, 3)

    # Vue d'ensemble avec skimpy (pour générer des statistiques détaillées)
    print("Résumé du DataFrame avec skimpy :")
    try:
        skim_result = skim(dataframe)  # Affiche un résumé général du DataFrame
        print(skim_result)
    except Exception as e:
        print(f"Skimpy a rencontré un problème : {e}")

    # Résumé des informations calculées dans un dictionnaire
    info_dataframe = {
        'Lignes': nb_lignes,
        'Colonnes': nb_colonnes,
        '% NaN': pourcentage_nan,
        '% Doublons': pourcentage_doublons,
        'Colonnes Objet': nb_colonnes_objet,
        'Colonnes Flottant': nb_colonnes_flottant,
        'Colonnes Entier': nb_colonnes_entier,
        'Colonnes Booléen': nb_colonnes_booleen,
        'Mémoire (Mo)': taille_memoire
    }

    # Convertit le dictionnaire en DataFrame pour une présentation claire sous forme de tableau
    return pd.DataFrame(info_dataframe, index=[0])


#### Résultat de l’Analyse Descriptive

La fonction `description_donnees` renvoie un tableau contenant :

- 📐 **Les dimensions et types de données du DataFrame.**
- ❓ **Les pourcentages de valeurs manquantes et de doublons.**
- 💾 **La mémoire totale utilisée.**

Cette synthèse permet d'identifier les points d'attention (doublons, valeurs manquantes) et d'évaluer la structure du DataFrame pour les prochaines étapes d'analyse.


In [11]:
# Appel de la fonction d'analyse descriptive
try:
    resultat_description = description_donnees(manga)

    # Affichage du résultat de l'analyse descriptive
    print("Résumé descriptif du DataFrame :")
    print(resultat_description)
except Exception as e:
    print(f"Erreur lors de l'exécution de la fonction : {e}")

Résumé du DataFrame avec skimpy :


╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 72975  │ │ string      │ 18    │                                                          │
│ │ Number of columns │ 30     │ │ float64     │ 6     │                                                          │
│ └───────────────────┴────────┘ │ int32       │ 4     │                                                          │
│                                │ bool        │ 2     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column_name     ┃ NA       ┃ NA %   ┃ mean    ┃ sd      ┃ p0    ┃ p25     ┃ p75      ┃ p100     ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ mal_id          │        0 │      0 │   87000 │   53000 │     1 │   34000 │   130000 │   180000 │  █▃▅▇▅▆  │  │
│ │ chapters        │    21000 │     29 │      22 │      55 │     1 │       5 │       20 │     6500 │    █     │  │
│ │ volumes         │    20000 │     28 │       3 │     4.7 │     1 │       1 │        3 │      200 │    █     │  │
│ │ score           │    46000 │     63 │     6.9 │    0.53 │   2.4 │     6.6 │      7.2 │      9.5 │     █▄   │  │
│ │ scored          │    46000 │     63 │     6.9 │    0.53 │   2.4 │     6.6 │      7.2 │      9.5 │     █▄   │  │
│ │ scored_by       │    46000 │     63 │    1800 │    9300 │   100 │     200 │     1100 │   420000 │    █     │  │
│ │ rank            │    22000 │     30 │   26000 │   16000 │     1 │   13000 │    41000 │    51000 │  ▆▆▆▅▄█  │  │
│ │ popularity      │        0 │      0 │   36000 │   21000 │     1 │   18000 │    55000 │    73000 │  ██████  │  │
│ │ members         │        0 │      0 │    1800 │   12000 │     1 │      67 │      790 │   730000 │    █     │  │
│ │ favorites       │        0 │      0 │      53 │    1100 │     0 │       0 │        3 │   130000 │    █     │  │
│ └─────────────────┴──────────┴────────┴─────────┴─────────┴───────┴─────────┴──────────┴──────────┴──────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column_name                   ┃ NA          ┃ NA %      ┃ words per row             ┃ total words          ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │ url                           │           0 │         0 │                         1 │                73000 │  │
│ │ images                        │           0 │         0 │                         1 │                73000 │  │
│ │ titles                        │           0 │         0 │                         1 │                73000 │  │
│ │ title                         │           0 │         0 │                         1 │                73000 │  │
│ │ title_english                 │       50000 │        69 │                         1 │                73000 │  │
│ │ title_japanese                │        1100 │       

None
Résumé descriptif du DataFrame :
   Lignes  Colonnes  % NaN  % Doublons  Colonnes Objet  Colonnes Flottant  Colonnes Entier  Colonnes Booléen  Mémoire (Mo)
0   72975        30  13.53        0.25              18                  6                4                 2        245.33


é><### 3.3 🔧 Nettoyage, Filtrage et Simplification des Données <a name="Nettoyage-Filtrage-Simplification-des-Données"></a>
   - 🔍 **Tri des Données** par `mal_id` pour un ordre cohérent.
   - ❌ **Suppression des Doublons** dans la colonne `mal_id` pour assurer l’unicité des enregistrements.
   - 🎯 **Filtrage par Type "Manga"** pour garder les données pertinentes.
   - 🗑️ **Suppression des Colonnes Non Essentielles** pour alléger le DataFrame.


In [12]:
# Tri du DataFrame en fonction de 'mal_id' par ordre croissant
manga.sort_values(by='mal_id', ascending=True, inplace=True)

# Suppression des doublons basée sur 'mal_id'
# Taille initiale du DataFrame avant la suppression des doublons
old_size = manga.shape[0]

# Suppression des doublons
manga = manga.drop_duplicates(subset=['mal_id']).reset_index(drop=True)

# Taille après suppression des doublons
new_size = manga.shape[0]
nb_doublons = old_size - new_size

# Affichage du nombre de doublons supprimés et aperçu du DataFrame nettoyé
print(f'Doublons supprimés : {nb_doublons}')
print("\nExtrait après la suppression des doublons :")
print(manga.head())

Doublons supprimés : 185

Extrait après la suppression des doublons :
   mal_id                                                url                                             images  approved                                             titles                    title                     title_english title_japanese  \
0       1            https://myanimelist.net/manga/1/Monster  {'jpg': {'image_url': 'https://cdn.myanimelist...      True  [{'type': 'Default', 'title': 'Monster'}, {'ty...                  Monster                           Monster        MONSTER   
1       2            https://myanimelist.net/manga/2/Berserk  {'jpg': {'image_url': 'https://cdn.myanimelist...      True  [{'type': 'Default', 'title': 'Berserk'}, {'ty...                  Berserk                           Berserk          ベルセルク   
2       3  https://myanimelist.net/manga/3/20th_Century_Boys  {'jpg': {'image_url': 'https://cdn.myanimelist...      True  [{'type': 'Default', 'title': '20th Century Bo...        

Dans cette section, nous filtrons les données pour ne conserver que les entrées de type **"Manga"**, car elles sont les plus pertinentes pour notre analyse. Nous supprimons également certaines colonnes non essentielles afin de simplifier le DataFrame et faciliter les traitements ultérieurs.

1. 🔍 **Filtrage par type "Manga"** : 
   - Nous affichons d’abord la répartition des différents types d'entrées pour voir les données disponibles.
   - Ensuite, nous appliquons un filtre pour ne garder que les entrées de type **"Manga"** et vérifions que le DataFrame ne contient plus que des mangas.

2. ❌ **Suppression des colonnes inutiles** : 
   - Les colonnes non pertinentes telles que **"publishing"**, **"scored"**, et **"explicit_genres"** sont supprimées pour alléger le DataFrame.
   - Un aperçu des colonnes avant et après suppression nous permet de valider la modification.

Grâce à ces opérations, nous obtenons un DataFrame plus cohérent et mieux adapté pour les analyses à venir.


In [13]:
# Affichage de la répartition des types avant le filtrage pour "Manga" uniquement
print("\nRépartition des types avant de filtrer par 'Manga' :")
print(manga['type'].value_counts())

# Filtrage pour conserver uniquement les entrées de type "Manga"
manga = manga[manga['type'] == 'Manga']

# Vérification de la répartition des types après le filtrage
print("\nRépartition des types après avoir filtré par 'Manga' :")
print(manga['type'].value_counts())

# Suppression des colonnes inutiles pour simplifier le DataFrame
# Affichage des colonnes avant suppression
print("\nColonnes avant suppression :")
print(manga.columns)

# Suppression des colonnes non pertinentes
manga.drop(columns=['publishing', 'scored', 'explicit_genres'], inplace=True)

# Vérification des colonnes après suppression
print("\nColonnes après suppression :")
print(manga.columns)


Répartition des types avant de filtrer par 'Manga' :
Manga          48812
Light Novel    10529
One-shot        6093
Manhwa          4371
Doujinshi       2323
Manhua           409
Novel            253
Name: type, dtype: Int64

Répartition des types après avoir filtré par 'Manga' :
Manga    48812
Name: type, dtype: Int64

Colonnes avant suppression :
Index(['mal_id', 'url', 'images', 'approved', 'titles', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'type', 'chapters', 'volumes', 'status', 'publishing', 'published', 'score', 'scored', 'scored_by', 'rank', 'popularity', 'members', 'favorites',
       'synopsis', 'background', 'authors', 'serializations', 'genres', 'explicit_genres', 'themes', 'demographics'],
      dtype='object')

Colonnes après suppression :
Index(['mal_id', 'url', 'images', 'approved', 'titles', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'type', 'chapters', 'volumes', 'status', 'published', 'score', 'scored_by', 'rank', 'popularity'

### 3.4 🔄 Transformation des Types de Données et des Champs Textuels <a name="Transformation-Types-Champs-Textuels"></a>

   - 🔄 **Conversion des Types de Données** pour optimiser la précision.
   - 📅 **Simplification des Champs de Dates** pour faciliter leur utilisation.
   - 🧹 **Nettoyage et Simplification des Champs Textuels** :
     - 🧼 Remplacement des valeurs non applicables (`None`, `n/a`) par `NaN`.
     - 🎨 **Simplification du champ `main_picture`** : Extraction des URLs pertinentes et remplacement des images par défaut par `NaN`.

In [14]:
# Afficher les 10 premières lignes de la colonne 'published' pour vérifier les données brutes
print(manga['published'].head(10))

# Afficher le nombre de types dans la colonne 'published' (e.g., str, dict) pour analyser sa structure
print(manga['published'].apply(type).value_counts())  

# Étape 1 : Extraction des dates à partir des chaînes JSON-like dans 'published'
# Utiliser une expression régulière pour extraire le texte entre "'from':" et "'to':" dans la chaîne
# Cela permet d'isoler les valeurs des dates de début ('from') et de fin ('to')
manga['published_from'] = manga['published'].str.extract(r"'from':\s*'([^']+)'")  # Extraction de la date 'from'
manga['published_to'] = manga['published'].str.extract(r"'to':\s*'([^']+)'")      # Extraction de la date 'to'

# Afficher les types de données extraits pour les colonnes 'published_from' et 'published_to'
# Permet de vérifier si certaines valeurs sont encore de type NA ou incorrectement extraites
print(manga['published_from'].apply(type).value_counts())  
print(manga['published_to'].apply(type).value_counts())  

# Étape 2 : Conversion des colonnes extraites en format datetime
# Convertir les chaînes extraites en type datetime, avec gestion des erreurs via 'coerce'
# Les erreurs (valeurs invalides) seront converties en NaT (Not a Time)
manga['published_from'] = pd.to_datetime(manga['published_from'], errors='coerce', utc=True).dt.date  # Conversion 'from'
manga['published_to'] = pd.to_datetime(manga['published_to'], errors='coerce', utc=True).dt.date      # Conversion 'to'

# Étape 3 : Affichage des résultats pour validation
# Vérifier les colonnes originales et extraites pour s'assurer de la cohérence des données
print(manga[['published', 'published_from', 'published_to']].head(10))

# Optionnel : Suppression de la colonne 'published' si elle n'est plus nécessaire
# Cela permet de garder un DataFrame plus propre pour l'analyse
manga.drop(columns=['published'], inplace=True)


0    {'from': '1994-12-05T00:00:00+00:00', 'to': '2...
1    {'from': '1989-08-25T00:00:00+00:00', 'to': No...
2    {'from': '1999-09-27T00:00:00+00:00', 'to': '2...
3    {'from': '1994-04-25T00:00:00+00:00', 'to': '2...
4    {'from': '1989-09-27T00:00:00+00:00', 'to': No...
5    {'from': '2001-12-01T00:00:00+00:00', 'to': '2...
6    {'from': '2003-05-21T00:00:00+00:00', 'to': '2...
7    {'from': '2003-02-24T00:00:00+00:00', 'to': '2...
8    {'from': '1999-09-21T00:00:00+00:00', 'to': '2...
9    {'from': '2001-08-07T00:00:00+00:00', 'to': '2...
Name: published, dtype: string
<class 'str'>    48812
Name: published, dtype: int64
<class 'str'>                            47544
<class 'pandas._libs.missing.NAType'>     1268
Name: published_from, dtype: int64
<class 'str'>                            31197
<class 'pandas._libs.missing.NAType'>    17615
Name: published_to, dtype: int64
                                           published published_from published_to
0  {'from': '1994-12-05T00:00

In [15]:
# Conversion des colonnes numériques pour éviter les floats inutiles
# En convertissant en 'Int64', on utilise un format qui accepte les valeurs manquantes sans transformer les entiers en float
for col in ['chapters', 'volumes', 'rank']:
    manga[col] = manga[col].astype('Int64')
    print(f"\nExemple de colonne '{col}' après conversion en Int64:")
    print(manga[col].head())

# Nettoyage des champs textuels
# Les valeurs non informatives dans 'synopsis' et 'background' sont remplacées par NaN pour un traitement des données plus propre
manga['synopsis'] = manga['synopsis'].replace(['', 'N/A', 'None.', '...'], np.nan)
manga['background'] = manga['background'].replace(['', 'N/A'], np.nan)

# Affichage pour vérifier le nettoyage
print("\nExtrait des colonnes 'synopsis' et 'background' après nettoyage :")
print(manga[['synopsis', 'background']].head(10))



Exemple de colonne 'chapters' après conversion en Int64:
0     162
1    <NA>
2     249
3     142
4    <NA>
Name: chapters, dtype: Int64

Exemple de colonne 'volumes' après conversion en Int64:
0      18
1    <NA>
2      22
3      14
4    <NA>
Name: volumes, dtype: Int64

Exemple de colonne 'rank' après conversion en Int64:
0     5
1     1
2    17
3    74
4    52
Name: rank, dtype: Int64

Extrait des colonnes 'synopsis' et 'background' après nettoyage :
                                            synopsis                                         background
0  Kenzou Tenma, a renowned Japanese neurosurgeon...  Monster won the Grand Prize at the third Tezuk...
1  Guts, a former mercenary now known as the Blac...  Berserk won the Excellence Award at the sixth ...
2  As the 20th century approaches its end, people...  20th Century Boys won the Kodansha Manga Award...
3  In a post-apocalyptic world where an environme...  Yokohama Kaidashi Kikou won the Seiun Award fo...
4  Makunouchi Ippo is 

Dans cette étape, nous appliquons des transformations spécifiques pour améliorer la qualité des données :

1. 🎨 **Simplification du champ `main_picture`** :
   - Le champ `main_picture` est extrait des URLs d'images dans le champ `images`.
   - Les images par défaut sont remplacées par `NaN` pour mieux gérer les valeurs manquantes et éviter les images non informatives.

2. 🧼 **Nettoyage des chaînes de caractères vides ou non applicables** :
   - Certaines colonnes textuelles contiennent des valeurs non significatives (comme `None`, `n/a`, etc.), qui sont converties en `NaN` pour garantir une meilleure cohérence des données.
   - Ce processus facilite l’analyse ultérieure en supprimant les valeurs textuelles inutiles ou génériques.


In [16]:
# Affichage des premières lignes de la colonne 'images'
print(manga['images'].head(10))
print(manga['images'].apply(type).value_counts())  # Vérifiez les types de données

0    {'jpg': {'image_url': 'https://cdn.myanimelist...
1    {'jpg': {'image_url': 'https://cdn.myanimelist...
2    {'jpg': {'image_url': 'https://cdn.myanimelist...
3    {'jpg': {'image_url': 'https://cdn.myanimelist...
4    {'jpg': {'image_url': 'https://cdn.myanimelist...
5    {'jpg': {'image_url': 'https://cdn.myanimelist...
6    {'jpg': {'image_url': 'https://cdn.myanimelist...
7    {'jpg': {'image_url': 'https://cdn.myanimelist...
8    {'jpg': {'image_url': 'https://cdn.myanimelist...
9    {'jpg': {'image_url': 'https://cdn.myanimelist...
Name: images, dtype: string
<class 'str'>    48812
Name: images, dtype: int64


In [17]:
# Convertir les chaînes en dictionnaires
def parse_images(row):
    try:
        # Utilise ast.literal_eval pour convertir les chaînes JSON-like en dictionnaires
        return ast.literal_eval(row)
    except (ValueError, SyntaxError):
        return None  # Retourne None en cas d'échec de la conversion

# Appliquer la conversion sur la colonne 'images'
manga['images_parsed'] = manga['images'].apply(parse_images)

# Extraire les URL des images principales
def extract_main_picture(image_dict):
    if isinstance(image_dict, dict):  # Vérifie si c'est un dictionnaire
        jpg_data = image_dict.get('jpg', {})  # Récupère la section 'jpg'
        return jpg_data.get('image_url', None)  # Récupère l'URL principale
    return None

manga['main_picture'] = manga['images_parsed'].apply(extract_main_picture)

# Vérifier les résultats
print("\nExtrait des URL principales :")
print(manga[['images', 'main_picture']].head())

manga.drop(columns=['images', 'images_parsed'], inplace=True)


Extrait des URL principales :
                                              images                                       main_picture
0  {'jpg': {'image_url': 'https://cdn.myanimelist...  https://cdn.myanimelist.net/images/manga/3/258...
1  {'jpg': {'image_url': 'https://cdn.myanimelist...  https://cdn.myanimelist.net/images/manga/1/157...
2  {'jpg': {'image_url': 'https://cdn.myanimelist...  https://cdn.myanimelist.net/images/manga/5/260...
3  {'jpg': {'image_url': 'https://cdn.myanimelist...  https://cdn.myanimelist.net/images/manga/1/171...
4  {'jpg': {'image_url': 'https://cdn.myanimelist...  https://cdn.myanimelist.net/images/manga/2/250...


In [18]:
# Nettoyage des chaînes de caractères vides ou non applicables
# Remplacement des valeurs non informatives (comme 'n/a' ou 'None') par NaN
columns_to_clean = ['type', 'synopsis', 'title_english', 'title_japanese']
replacements = {'': np.nan, 'n/a': np.nan, 'None': np.nan, 'N/A': np.nan}

for col in columns_to_clean:
    print(f"\nExtrait de '{col}' avant nettoyage :")
    print(manga[col].head())
    manga[col] = manga[col].replace(replacements)
    print(f"\nExtrait de '{col}' après nettoyage :")
    print(manga[col].head())



Extrait de 'type' avant nettoyage :
0    Manga
1    Manga
2    Manga
3    Manga
4    Manga
Name: type, dtype: string

Extrait de 'type' après nettoyage :
0    Manga
1    Manga
2    Manga
3    Manga
4    Manga
Name: type, dtype: string

Extrait de 'synopsis' avant nettoyage :
0    Kenzou Tenma, a renowned Japanese neurosurgeon...
1    Guts, a former mercenary now known as the Blac...
2    As the 20th century approaches its end, people...
3    In a post-apocalyptic world where an environme...
4    Makunouchi Ippo is a 16-year-old high school s...
Name: synopsis, dtype: string

Extrait de 'synopsis' après nettoyage :
0    Kenzou Tenma, a renowned Japanese neurosurgeon...
1    Guts, a former mercenary now known as the Blac...
2    As the 20th century approaches its end, people...
3    In a post-apocalyptic world where an environme...
4    Makunouchi Ippo is a 16-year-old high school s...
Name: synopsis, dtype: string

Extrait de 'title_english' avant nettoyage :
0                         

### 3.5 👤 Transformation des Auteurs <a name="Transformation-des-Auteurs"></a>

Dans cette étape, nous procédons à la séparation des auteurs en deux catégories distinctes : **auteurs individuels** et **collectifs**. 

- **Auteurs individuels** : Les auteurs sont considérés comme individuels lorsque leur nom est sous le format "Nom, Prénom". Cela permet de distinguer les contributions spécifiques de chaque auteur.
  
- **Collectifs** : Les collectifs regroupent les studios, équipes, ou noms d'auteurs sans format "Nom, Prénom", permettant ainsi de mieux différencier les contributions collectives ou celles qui ne sont pas attribuées à un auteur unique.

Cette distinction améliore la qualité de l'analyse en facilitant des recherches et des regroupements précis par type de contributeurs.


In [19]:
# Fonction pour convertir les chaînes JSON-like en objets Python
def parse_authors(row):
    if isinstance(row, str):  # Vérifie si c'est une chaîne
        try:
            return ast.literal_eval(row)  # Convertit en liste de dictionnaires
        except (ValueError, SyntaxError):
            return None  # Retourne None en cas d'échec
    return row  # Si ce n'est pas une chaîne, retourne tel quel

# Appliquer la conversion sur la colonne 'authors'
manga['authors'] = manga['authors'].apply(parse_authors)

# Vérification après parsing
print("\nExtrait des données après parsing :")
print(manga['authors'].head(10))
print(manga['authors'].apply(type).value_counts())  # Vérifiez les types de données

# Vérifiez si plusieurs auteurs ou collectifs sont présents et formatez les résultats comme listes
def author_format(authors: List[dict]) -> Tuple[List[dict], List[dict]]:
    individual_authors, collectives = [], []
    for author in authors or []:
        mal_id, name = author.get('mal_id'), author.get('name')
        if ',' in name:
            # Séparation des noms pour les auteurs individuels
            last_name, first_name = name.split(', ', 1)
            individual_authors.append({'id': mal_id, 'first_name': first_name, 'last_name': last_name})
        else:
            # Traitement des noms comme collectifs pour les studios ou noms uniques
            collectives.append({'id': mal_id, 'name': name})
    return individual_authors, collectives

# Appliquer la fonction pour séparer les auteurs individuels et collectifs
manga = manga.assign(
    individual_authors=manga['authors'].apply(lambda x: author_format(x)[0]),
    collectives=manga['authors'].apply(lambda x: author_format(x)[1])
)

# Affichage des résultats sous forme de listes lisibles
print("--- Auteurs individuels après traitement ---")
for index, row in manga['individual_authors'].head(10).items():
    print(f"Index {index}: {row}")

print("\n--- Collectifs après traitement ---")
for index, row in manga['collectives'].head(10).items():
    print(f"Index {index}: {row}")



Extrait des données après parsing :
0    [{'mal_id': 1867, 'type': 'people', 'name': 'U...
1    [{'mal_id': 1868, 'type': 'people', 'name': 'M...
2    [{'mal_id': 1867, 'type': 'people', 'name': 'U...
3    [{'mal_id': 1869, 'type': 'people', 'name': 'A...
4    [{'mal_id': 1876, 'type': 'people', 'name': 'M...
5    [{'mal_id': 1878, 'type': 'people', 'name': 'T...
6    [{'mal_id': 1877, 'type': 'people', 'name': 'C...
7    [{'mal_id': 1877, 'type': 'people', 'name': 'C...
8    [{'mal_id': 1879, 'type': 'people', 'name': 'K...
9    [{'mal_id': 1880, 'type': 'people', 'name': 'K...
Name: authors, dtype: object
<class 'list'>    48812
Name: authors, dtype: int64
--- Auteurs individuels après traitement ---
Index 0: [{'id': 1867, 'first_name': 'Naoki', 'last_name': 'Urasawa'}]
Index 1: [{'id': 1868, 'first_name': 'Kentarou', 'last_name': 'Miura'}]
Index 2: [{'id': 1867, 'first_name': 'Naoki', 'last_name': 'Urasawa'}]
Index 3: [{'id': 1869, 'first_name': 'Hitoshi', 'last_name': 'Ashinano'}]

In [20]:
# Créer des colonnes lisibles pour les noms
manga['individual_authors_names'] = manga['individual_authors'].apply(
    lambda authors: ', '.join(f"{a['first_name']} {a['last_name']}" for a in authors) if authors else np.nan
)
manga['collectives_names'] = manga['collectives'].apply(
    lambda collectives: ', '.join(c['name'] for c in collectives) if collectives else np.nan
)

# Affichage des nouvelles colonnes pour vérifier le traitement
print("--- Noms des auteurs individuels ---")
print(manga[['individual_authors', 'individual_authors_names']].head(10))

print("\n--- Noms des collectifs ---")
print(manga[['collectives', 'collectives_names']].head(10))

# Remplacer les listes vides par NaN dans les colonnes d'auteurs si nécessaire
manga['individual_authors'] = manga['individual_authors'].apply(lambda x: x if x else np.nan)
manga['collectives'] = manga['collectives'].apply(lambda x: x if x else np.nan)

# Vérification finale des colonnes principales
print("\n--- Résumé des colonnes principales après transformation ---")
print(manga[['individual_authors', 'individual_authors_names', 'collectives', 'collectives_names']].head(10))


--- Noms des auteurs individuels ---
                                  individual_authors individual_authors_names
0  [{'id': 1867, 'first_name': 'Naoki', 'last_nam...            Naoki Urasawa
1  [{'id': 1868, 'first_name': 'Kentarou', 'last_...           Kentarou Miura
2  [{'id': 1867, 'first_name': 'Naoki', 'last_nam...            Naoki Urasawa
3  [{'id': 1869, 'first_name': 'Hitoshi', 'last_n...         Hitoshi Ashinano
4  [{'id': 1876, 'first_name': 'George', 'last_na...          George Morikawa
5  [{'id': 1878, 'first_name': 'Arina', 'last_nam...           Arina Tanemura
6                                                 []                      NaN
7                                                 []                      NaN
8  [{'id': 1879, 'first_name': 'Masashi', 'last_n...        Masashi Kishimoto
9  [{'id': 1880, 'first_name': 'Tite', 'last_name...                Tite Kubo

--- Noms des collectifs ---
                              collectives collectives_names
0               

In [21]:
# Supprimer les colonnes inutiles après transformation
manga.drop(columns=['individual_authors', 'collectives', 'authors'], inplace=True)

# Affichage pour vérifier les colonnes restantes
print("--- Résumé des colonnes après suppression ---")
print(manga[['individual_authors_names', 'collectives_names']].head(10))

--- Résumé des colonnes après suppression ---
  individual_authors_names collectives_names
0            Naoki Urasawa               NaN
1           Kentarou Miura       Studio Gaga
2            Naoki Urasawa               NaN
3         Hitoshi Ashinano               NaN
4          George Morikawa               NaN
5           Arina Tanemura               NaN
6                      NaN             CLAMP
7                      NaN             CLAMP
8        Masashi Kishimoto               NaN
9                Tite Kubo               NaN


### 3.6 🏷️ Transformation des Catégories <a name="Transformation-des-Catégories"></a>

Les colonnes de catégories telles que 'serializations', 'genres', 'themes', 'demographics', et 'title_synonyms' sont traitées pour ne conserver que les noms. Cette étape simplifie ces colonnes, facilitant ainsi les analyses ultérieures.

1. **Vérification des Données**  
   Avant d'effectuer les transformations, on examine la structure des données dans chaque colonne pour s'assurer que le format est cohérent. Cette vérification permet d'éviter les erreurs lors des opérations de nettoyage.

2. **Transformation des Colonnes**  
   Pour chaque colonne, si les éléments sont des listes de dictionnaires, nous conservons uniquement les valeurs de la clé 'name'. Dans les autres cas, si les éléments sont des listes de chaînes, nous les fusionnons en une seule chaîne séparée par des virgules.

3. **Définition de la Catégorie R18+**  
   Pour identifier les mangas de contenu NSFW (Not Safe For Work), une nouvelle colonne 'nsfw' est ajoutée, marquant les œuvres avec les genres 'Hentai' ou 'Erotica'.


In [22]:
# Affichage des premières lignes de chaque colonne pour vérifier leur contenu
columns_to_check = ['serializations', 'genres', 'themes', 'demographics', 'title_synonyms']

for col in columns_to_check:
    print(f"\n--- Vérification du contenu pour la colonne '{col}' ---")
    print("Premier élément :", manga[col].iloc[0])
    print("Type :", type(manga[col].iloc[0]))
    print("Extrait des 5 premières lignes :")
    print(manga[col].head())



--- Vérification du contenu pour la colonne 'serializations' ---
Premier élément : [{'mal_id': 1, 'type': 'manga', 'name': 'Big Comic Original', 'url': 'https://myanimelist.net/manga/magazine/1/Big_Comic_Original'}]
Type : <class 'str'>
Extrait des 5 premières lignes :
0    [{'mal_id': 1, 'type': 'manga', 'name': 'Big C...
1    [{'mal_id': 2, 'type': 'manga', 'name': 'Young...
2    [{'mal_id': 3, 'type': 'manga', 'name': 'Big C...
3    [{'mal_id': 4, 'type': 'manga', 'name': 'After...
4    [{'mal_id': 8, 'type': 'manga', 'name': 'Shoun...
Name: serializations, dtype: string

--- Vérification du contenu pour la colonne 'genres' ---
Premier élément : [{'mal_id': 46, 'type': 'manga', 'name': 'Award Winning', 'url': 'https://myanimelist.net/manga/genre/46/Award_Winning'}, {'mal_id': 8, 'type': 'manga', 'name': 'Drama', 'url': 'https://myanimelist.net/manga/genre/8/Drama'}, {'mal_id': 7, 'type': 'manga', 'name': 'Mystery', 'url': 'https://myanimelist.net/manga/genre/7/Mystery'}]
Type : <cl

In [23]:
# Étape 1 : Fonction pour transformer une colonne contenant des listes de dictionnaires en une chaîne de noms
def extract_names(column):
    """
    Transforme une colonne contenant des chaînes de listes JSON ou des listes de dictionnaires 
    en une chaîne contenant uniquement les valeurs de la clé 'name', séparées par des virgules.

    Arguments:
        column (pd.Series): La colonne à transformer.

    Retourne:
        pd.Series: La colonne transformée.
    """
    def process_entry(entry):
        if isinstance(entry, str):
            try:
                # Convertit la chaîne en liste de dictionnaires
                parsed = ast.literal_eval(entry)
                if isinstance(parsed, list):
                    return ', '.join(dic['name'] for dic in parsed if isinstance(dic, dict) and 'name' in dic)
            except (ValueError, SyntaxError):
                # Si la conversion échoue, retourne l'entrée originale
                return entry
        elif isinstance(entry, list):
            # Si c'est déjà une liste, extrait les noms
            return ', '.join(dic['name'] for dic in entry if isinstance(dic, dict) and 'name' in dic)
        return entry  # Retourne l'entrée originale si aucun traitement n'est applicable
    
    return column.apply(process_entry)

# Étape 2 : Appliquer cette transformation aux colonnes cibles
columns_to_transform = ['serializations', 'genres', 'themes', 'demographics', 'title_synonyms']
for col in columns_to_transform:
    print(f"\n--- Transformation de la colonne '{col}' ---")
    print("Avant transformation :", manga[col].head(3).to_list())
    manga[col] = extract_names(manga[col])
    print("Après transformation :", manga[col].head(3).to_list())

# Étape 3 : Vérification des colonnes après transformation
print("\n--- Vérification des colonnes après transformation ---")
verification_df = manga[columns_to_transform].head(5)
print(verification_df)

# Étape 4 : Création de la colonne 'nsfw' pour identifier les contenus adultes
manga['nsfw'] = manga['genres'].apply(
    lambda genres: any(genre.strip() in ['Hentai', 'Erotica'] for genre in genres.split(',')) if isinstance(genres, str) else False
)

# Étape 5 : Vérification de la colonne 'nsfw'
print("\n--- Vérification de la colonne 'nsfw' ---")
print(manga[['genres', 'nsfw']].head(5))



--- Transformation de la colonne 'serializations' ---
Avant transformation : ["[{'mal_id': 1, 'type': 'manga', 'name': 'Big Comic Original', 'url': 'https://myanimelist.net/manga/magazine/1/Big_Comic_Original'}]", "[{'mal_id': 2, 'type': 'manga', 'name': 'Young Animal', 'url': 'https://myanimelist.net/manga/magazine/2/Young_Animal'}]", "[{'mal_id': 3, 'type': 'manga', 'name': 'Big Comic Spirits', 'url': 'https://myanimelist.net/manga/magazine/3/Big_Comic_Spirits'}]"]
Après transformation : ['Big Comic Original', 'Young Animal', 'Big Comic Spirits']

--- Transformation de la colonne 'genres' ---
Avant transformation : ["[{'mal_id': 46, 'type': 'manga', 'name': 'Award Winning', 'url': 'https://myanimelist.net/manga/genre/46/Award_Winning'}, {'mal_id': 8, 'type': 'manga', 'name': 'Drama', 'url': 'https://myanimelist.net/manga/genre/8/Drama'}, {'mal_id': 7, 'type': 'manga', 'name': 'Mystery', 'url': 'https://myanimelist.net/manga/genre/7/Mystery'}]", "[{'mal_id': 1, 'type': 'manga', 'name

### 3.7 📊 Aperçu détaillé du DataFrame `manga` <a name="Aperçu-détaillé-du-DataFrame"></a>


Pour obtenir une vue d'ensemble complète du DataFrame `manga`, nous utilisons la bibliothèque `skimpy` pour générer un résumé global, incluant des informations détaillées sur les types de données, la distribution des valeurs manquantes, et d'autres statistiques utiles. Cette vue nous aide à identifier les colonnes qui pourraient nécessiter des traitements additionnels avant d'entamer les analyses ou les prédictions. 

En complément, nous affichons également les premières et dernières lignes du DataFrame pour visualiser un extrait concret des données et mieux comprendre leur structure et contenu.


In [24]:
# Création d'une copie temporaire du DataFrame pour le résumé avec skimpy
# Conversion des colonnes de dates en chaînes et remplacement des NA pour les colonnes entières
manga_skim = manga.copy()
date_columns = manga.select_dtypes(include=["datetime", "object"]).columns
manga_skim[date_columns] = manga[date_columns].astype(str)
manga_skim = manga_skim.fillna(value={col: -1 for col in manga.select_dtypes(include="Int64").columns})

# Aperçu global du DataFrame `manga` avec skimpy
print("Aperçu global du DataFrame `manga` :")
skim(manga_skim)

# Affichage des 5 premières lignes du DataFrame
print("\nPremières lignes du DataFrame :")
print(manga.head())

# Affichage des 5 dernières lignes du DataFrame
print("\nDernières lignes du DataFrame :")
print(manga.tail())


Aperçu global du DataFrame `manga` :


╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 48812  │ │ string      │ 19    │                                                          │
│ │ Number of columns │ 30     │ │ int32       │ 7     │                                                          │
│ └───────────────────┴────────┘ │ bool        │ 2     │                                                          │
│                                │ float64     │ 2     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column_name     ┃ NA       ┃ NA %   ┃ mean    ┃ sd      ┃ p0    ┃ p25     ┃ p75      ┃ p100     ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ mal_id          │        0 │      0 │   81000 │   55000 │     1 │   23000 │   130000 │   180000 │  █▃▃▅▅▄  │  │
│ │ chapters        │        0 │      0 │      17 │      50 │    -1 │      -1 │       16 │     6500 │    █     │  │
│ │ volumes         │        0 │      0 │     2.4 │     4.8 │    -1 │       1 │        3 │      200 │    █     │  │
│ │ score           │    28000 │     57 │     6.9 │    0.51 │   2.4 │     6.6 │      7.2 │      9.5 │     █▄   │  │
│ │ scored_by       │    28000 │     57 │    2100 │   10000 │   100 │     220 │     1200 │   420000 │    █     │  │
│ │ rank            │        0 │      0 │   18000 │   17000 │    -1 │      -1 │    32000 │    51000 │  █▃▃▂▂▃  │  │
│ │ popularity      │        0 │      0 │   33000 │   21000 │     1 │   15000 │    50000 │    73000 │  █▇▇▆▅▅  │  │
│ │ members         │        0 │      0 │    2300 │   14000 │     1 │      83 │     1100 │   730000 │    █     │  │
│ │ favorites       │        0 │      0 │      71 │    1300 │     0 │       0 │        4 │   130000 │    █     │  │
│ └─────────────────┴──────────┴────────┴─────────┴─────────┴───────┴─────────┴──────────┴──────────┴──────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column_name                         ┃ NA         ┃ NA %     ┃ words per row           ┃ total words        ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩  │
│ │ url                                 │          0 │        0 │                       1 │              49000 │  │
│ │ titles                              │          0 │        0 │                       1 │              49000 │  │
│ │ title                               │          0 │        0 │                       1 │              49000 │  │
│ │ title_english                       │      32000 │       66 │                       1 │              49000 │  │
│ │ title_japanese                      │        580 │      1.2 │                       1 │              49000 │  │
│ │ title_synonyms                      │          0 │        0 │                       1 │              49000 │  │
│ │ type                                │          0 │  


Premières lignes du DataFrame :
   mal_id                                                url  approved                                             titles                    title                     title_english title_japanese title_synonyms   type  chapters  volumes      status  score  scored_by  rank  \
0       1            https://myanimelist.net/manga/1/Monster      True  [{'type': 'Default', 'title': 'Monster'}, {'ty...                  Monster                           Monster        MONSTER                 Manga       162       18    Finished   9.16   104452.0     5   
1       2            https://myanimelist.net/manga/2/Berserk      True  [{'type': 'Default', 'title': 'Berserk'}, {'ty...                  Berserk                           Berserk          ベルセルク                 Manga      <NA>     <NA>  Publishing   9.47   364096.0     1   
2       3  https://myanimelist.net/manga/3/20th_Century_Boys      True  [{'type': 'Default', 'title': '20th Century Bo...        20th Centu

### Résultats de l'Aperçu

Grâce à cette analyse combinée, nous obtenons une compréhension à la fois macro et micro du DataFrame `manga`. Le résumé global nous montre les caractéristiques générales de chaque colonne, tandis que les extraits des premières et dernières lignes offrent une illustration concrète des valeurs présentes. Ces informations permettent de vérifier la qualité des données, leur cohérence et leur complétude avant d'aller plus loin dans l'analyse ou la modélisation.


## 📊 4. Analyse Exploratoire des Données <a name="Analyse-Exploratoire-des-Données"></a>

Une analyse de base est effectuée pour tirer des insights des données. Cela peut inclure l'étude de la popularité des mangas, la distribution des genres, la fréquence des publications, etc.


### 4.1 📈 Analyse de la Popularité et Distribution des Scores <a name="Analyse-de-la-Popularité-et-Distribution-des-Scores"></a>

Dans cette section, nous examinons les scores des mangas pour obtenir une vue d’ensemble de leur popularité. Nous allons afficher des statistiques descriptives sur les scores, puis créer une visualisation de la distribution des scores afin de repérer les mangas les mieux notés et la tendance générale des évaluations.

In [25]:
# Analyse de la Popularité et du Score
print("### Analyse de la Popularité et du Score")
print("Description statistique des scores des mangas:\n", manga['score'].describe())


### Analyse de la Popularité et du Score
Description statistique des scores des mangas:
 count    20778.000000
mean         6.918589
std          0.505704
min          2.430000
25%          6.600000
50%          6.890000
75%          7.210000
max          9.470000
Name: score, dtype: float64


In [26]:
# Création de l'histogramme interactif de la distribution des scores avec Plotly Express
fig = px.histogram(manga, x='score', nbins=20, 
                   title='Distribution des Scores des Mangas',
                   color_discrete_sequence=["#2ca02c"])  # Couleur verte personnalisée
fig.show()

### 4.2 📆 Fréquence des Publications par Année <a name="Fréquence-des-Publications-par-Année"></a>

Pour mieux comprendre l’évolution du marché des mangas, nous analysons la fréquence de publication par année. Cette analyse nous permet de voir les périodes les plus prolifiques dans l’histoire des mangas.

In [27]:
# Analyse de la Fréquence des Publications
print("### Analyse de la Fréquence des Publications")
manga['published_year'] = pd.to_datetime(manga['published_from']).dt.year
 
# Comptage des publications par année
publications_per_year = manga['published_year'].value_counts().sort_index()

# Création du graphique linéaire interactif
fig = px.line(x=publications_per_year.index, y=publications_per_year.values, 
              title="Fréquence des Publications de Mangas par Année")

fig.update_layout(xaxis_title="Année", yaxis_title="Nombre de Publications")

fig.show()


### Analyse de la Fréquence des Publications


### 4.3 ✍️ Contributions des Auteurs Individuels et Collectifs <a name="Contributions-des-Auteurs-Individuels-et-Collectifs"></a>

Cette analyse permet de distinguer les contributions des auteurs individuels de celles des collectifs ou studios. Cela donne une perspective sur la diversité des créateurs impliqués dans l’industrie du manga.

In [28]:
# Contributions des Auteurs Individuels et Collectifs
print("### Contributions des Auteurs Individuels et Collectifs")
print("Nombre d'oeuvres par auteurs individuels :", len(manga['individual_authors_names'].explode().dropna()))
print("Nombre d'oeuvres par collectifs :", len(manga['collectives_names'].explode().dropna()))

### Contributions des Auteurs Individuels et Collectifs
Nombre d'oeuvres par auteurs individuels : 41080
Nombre d'oeuvres par collectifs : 10248


In [29]:
# Calcul du nombre d'œuvres par type d'auteur
individual_authors_count = len(manga['individual_authors_names'].explode().dropna())
collectives_count = len(manga['collectives_names'].explode().dropna())

# Création d'un DataFrame pour Plotly
author_data = pd.DataFrame({
    "Type d'Auteur": ["Individuel", "Collectif"],
    "Nombre de Publications": [individual_authors_count, collectives_count]
})

# Création du graphique à barres interactif
fig = px.bar(
    author_data, 
    x="Type d'Auteur", 
    y="Nombre de Publications", 
    title="Contributions des Auteurs Individuels vs Collectifs",
    color_discrete_sequence=px.colors.qualitative.Pastel  # Palette pastel
)
fig.show()

### 4.4 💥 Analyse des Thèmes et Démographies <a name="Analyse-des-Thèmes-et-Démographies"></a>


Pour explorer les thèmes et les audiences visées dans les mangas, nous analysons la répartition des thèmes et des démographies. Cette analyse aide à identifier les préférences thématiques des lecteurs et les groupes démographiques ciblés.

In [30]:
# Analyse des Thèmes et Démographies

# Comptage des thèmes les plus populaires, en excluant les valeurs manquantes ou vides
theme_counts = (manga['themes']
                .dropna()  # Supprime les valeurs NaN
                .str.split(', ')  # Divise les thèmes en liste
                .explode()  # Transforme la liste en une seule colonne
                .str.strip()  # Supprime les espaces autour des valeurs
                .replace('', np.nan)  # Remplace les chaînes vides par NaN
                .dropna()  # Supprime à nouveau les NaN après nettoyage
                .value_counts()  # Comptage des occurrences
                .head(10)  # Garde les 10 plus fréquents
                .reset_index())  # Réinitialise l'index
theme_counts.columns = ['Thème', 'Occurrences']

# Création du graphique à barres interactif
fig = px.bar(theme_counts, x='Occurrences', y='Thème', 
             title='Top 10 des Thèmes les Plus Populaires (Sans Valeurs Manquantes)',
             orientation='h', color='Occurrences', 
             color_continuous_scale=px.colors.sequential.Plasma)

fig.update_layout(xaxis_title='Nombre d’Occurrences', yaxis_title='Thème')
fig.show()


In [31]:
# Comptage des démographies ciblées, en excluant les valeurs manquantes ou vides
demographic_counts = (manga['demographics']
                      .dropna()  # Supprime les valeurs NaN
                      .str.split(', ')  # Divise les démographies en liste
                      .explode()  # Transforme la liste en une seule colonne
                      .str.strip()  # Supprime les espaces autour des valeurs
                      .replace('', np.nan)  # Remplace les chaînes vides par NaN
                      .dropna()  # Supprime à nouveau les NaN après nettoyage
                      .value_counts()  # Comptage des occurrences
                      .reset_index())  # Réinitialise l'index
demographic_counts.columns = ['Démographie', 'Occurrences']

# Création du graphique circulaire interactif
fig = px.pie(demographic_counts, values='Occurrences', names='Démographie',
             title='Répartition des Démographies Ciblées (Sans Valeurs Manquantes)',
             hole=0.3)  # Option pour créer un graphique en donut

fig.show()


### 4.5 🔞 Proportion de Mangas NSFW <a name="Proportion-de-Mangas-NSFW"></a>

Nous examinons ici la proportion de mangas classés comme NSFW (Not Safe For Work), incluant les genres Hentai et Erotica, pour évaluer l’orientation du contenu.

In [32]:
# Proportion de Mangas NSFW
nsfw_count = manga['nsfw'].value_counts(normalize=True) * 100

# Création d'un DataFrame pour Plotly
nsfw_data = nsfw_count.reset_index()
nsfw_data.columns = ['NSFW', 'Proportion']

# Création du graphique circulaire interactif
fig = px.pie(nsfw_data, values='Proportion', names='NSFW',
             title='Proportion de Mangas NSFW',
             labels={'NSFW': 'Type de Manga'},
             color='Proportion',
             color_discrete_sequence=['#66b3ff', '#ff9999'])

fig.update_traces(textinfo='percent+label')  # Afficher pourcentage et label dans la bulle

fig.show()

### 4.6 🔍 Vérification des Types de Données et des Valeurs Manquantes <a name="Vérification-des-Types-de-Données-et-Valeurs-Manquantes"></a>

Nous vérifions les types de données de chaque colonne, recherchons les valeurs manquantes et affichons des statistiques descriptives pour s’assurer de la qualité du DataFrame.

In [33]:
# 1. Aperçu du DataFrame
print("### Aperçu du DataFrame")
print(manga.head(2))  # Afficher les 2 premières lignes du DataFrame
print("\n")
print(manga.tail(2))  # Afficher les 2 dernières lignes du DataFrame

# 2. Tableau des Types de Données
data_types = manga.dtypes.astype(str).reset_index()  # Conversion en chaînes
data_types.columns = ['Colonne', 'Type de Données']

# Création du tableau
fig_types = go.Figure(data=[go.Table(
    header=dict(values=['Colonne', 'Type de Données']),
    cells=dict(values=[data_types['Colonne'], data_types['Type de Données']]))
])

fig_types.update_layout(title='Types de Données dans le DataFrame')
fig_types.show()

# 3. Graphique des Valeurs Manquantes
missing_counts = manga.isna().sum()
missing_data = missing_counts[missing_counts > 0].reset_index()
missing_data.columns = ['Colonne', 'Valeurs Manquantes']

# Création du graphique à barres pour les valeurs manquantes
fig_missing = px.bar(missing_data, x='Colonne', y='Valeurs Manquantes',
                     title='Valeurs Manquantes dans chaque Colonne',
                     color='Valeurs Manquantes',
                     color_continuous_scale=px.colors.sequential.Viridis)

fig_missing.update_layout(xaxis_title='Colonne', yaxis_title='Nombre de Valeurs Manquantes')
fig_missing.show()

# 4. Statistiques Descriptives
print("### Statistiques Descriptives")
print(manga.describe())


### Aperçu du DataFrame
   mal_id                                      url  approved                                             titles    title title_english title_japanese title_synonyms   type  chapters  volumes      status  score  scored_by  rank  popularity  members  favorites  \
0       1  https://myanimelist.net/manga/1/Monster      True  [{'type': 'Default', 'title': 'Monster'}, {'ty...  Monster       Monster        MONSTER                 Manga       162       18    Finished   9.16   104452.0     5          29   258877      22021   
1       2  https://myanimelist.net/manga/2/Berserk      True  [{'type': 'Default', 'title': 'Berserk'}, {'ty...  Berserk       Berserk          ベルセルク                 Manga      <NA>     <NA>  Publishing   9.47   364096.0     1           1   725789     130593   

                                            synopsis                                         background      serializations                                             genres               

### Statistiques Descriptives
              mal_id   chapters   volumes         score      scored_by          rank    popularity        members      favorites  published_year
count   48812.000000    36160.0   40307.0  20778.000000   20778.000000       35059.0  48812.000000   48812.000000   48812.000000    47544.000000
mean    80957.920061  23.021681  3.117945      6.918589    2058.762634  25022.198694  33277.247480    2290.945218      70.930468     2010.564088
std     55295.745295  56.365601  4.976049      0.505704   10298.209301    15560.4881  20891.894499   13657.012009    1284.661703        9.946466
min         1.000000        1.0       1.0      2.430000     100.000000           1.0      1.000000       1.000000       0.000000     1928.000000
25%     22725.500000        6.0       1.0      6.600000     218.000000       11552.0  15058.750000      83.000000       0.000000     2006.000000
50%     88720.000000       10.0       1.0      6.890000     459.000000       22844.0  31746.500000  

## 💾 5. Exportation des Données <a name="Exportation-des-Données"></a>

Après la préparation et l'analyse, nous exportons le DataFrame final pour conserver les données nettoyées et transformées.

- **Exportation** : Les données sont sauvegardées en format CSV ou JSON pour faciliter leur utilisation ultérieure.
- **Vérification de l'Exportation** : Nous vérifions les premières lignes du fichier exporté pour s’assurer de son intégrité.


In [34]:
# Définir les chemins d'exportation pour le fichier principal et pour les liens
output_path_main = notebook_config["data_directory"] / f"{notebook_config['csv_output_file']}_data.csv"
output_path_links = notebook_config["data_directory"] / f"{notebook_config['csv_output_file']}_links.csv"

# Séparer les colonnes de liens du DataFrame principal
manga_links = manga[['mal_id', 'url', 'main_picture']]
manga_main = manga.drop(columns=['url', 'main_picture'])

# Exporter le DataFrame principal sans les liens au format CSV
try:
    manga_main.to_csv(output_path_main, index=False)
    logger.info(f"Le DataFrame principal a été enregistré dans : {output_path_main}")
except Exception as e:
    logger.error(f"Erreur lors de l'enregistrement du DataFrame principal : {e}")

# Exporter les liens dans un fichier CSV
try:
    manga_links.to_csv(output_path_links, index=False)
    logger.info(f"Les liens ont été enregistrés dans un fichier CSV à : {output_path_links}")
except Exception as e:
    logger.error(f"Erreur lors de l'enregistrement des liens : {e}")


### 📜  6. Conclusion <a name="Conclusion"></a>

Dans ce notebook, nous avons mené une analyse approfondie de la base de données de mangas issue de l'API Jikan, en suivant un workflow structuré et complet. Voici un résumé des étapes essentielles que nous avons accomplies :

1. **Scraping des Données**  
   - Configuration et exécution du scraping via l'API Jikan.
   - Collecte des données page par page, sauvegarde dans des fichiers JSON structurés.
   - Fusion des fichiers JSON en une base unifiée pour faciliter l’analyse.

2. **Chargement et Préparation des Données**  
   - Chargement des données fusionnées dans un DataFrame.
   - Nettoyage et transformation des données pour corriger les incohérences.
   - Structuration des informations pour une meilleure exploitabilité.

3. **Analyse Exploratoire des Données**  
   - Exploration des tendances clés à l’aide de visualisations interactives et de statistiques descriptives.  
   - Analyse des thèmes et genres les plus populaires.  
   - Étude des contributions des auteurs individuels et collectifs.  
   - Découverte des préférences démographiques liées aux œuvres.

4. **Exportation des Données**  
   - Création d’un DataFrame final, nettoyé et enrichi.
   - Exportation pour faciliter des analyses futures ou des intégrations dans d’autres projets.

### Perspectives Futures  
Les données préparées dans ce projet offrent des opportunités variées, notamment :  
- **Modèles prédictifs** pour évaluer la popularité future des mangas.  
- **Systèmes de recommandation personnalisés** basés sur les préférences des utilisateurs.  
- **Visualisations avancées** pour découvrir de nouvelles tendances dans l’industrie du manga.  

En conclusion, ce projet fournit une base solide pour approfondir l’analyse des mangas et explorer des applications innovantes autour de ces données.
